In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / "src"))
from target import build_metabolic_target

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score

from sklearn.inspection import permutation_importance
%matplotlib inline

In [3]:
DATA_PATH = Path("data/processed/nhanes_clean.csv")

In [4]:
df = pd.read_csv(DATA_PATH)
df = df[df["age"] >= 20]
df["metabolic_syndrome"] = build_metabolic_target(df)
df = df.dropna(subset=["metabolic_syndrome"])
sample_weights = df["weight_mec_2yr"]
drop_cols = [
    "uid",
    "SEQN",
    "cycle",
    "weight_mec_2yr",
    "weight_int_2yr",
    "svy_psu",
    "svy_stratum",
    "waist_cm",
    "bp_systolic",
    "bp_diastolic",
    "triglycerides",
    "hdl",
    "hba1c",
    "metabolic_syndrome",
    "bmi",
    "total_chol",
    "ldl",
    "race_eth",
    "education",
    "income_poverty_ratio",
    "food_security_adult"
]

x = df.drop(columns=drop_cols)
y = df["metabolic_syndrome"]

x_train, x_test, y_train, y_test, w_train, w_test = train_test_split(
    x,
    y,
    sample_weights,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_split=50 ,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
)
rf.fit(x_train, y_train, sample_weight=w_train)

# Performance
y_train_pred = rf.predict(x_train)
y_test_pred = rf.predict(x_test)
print("Training Performance")
print(classification_report(y_train, y_train_pred, sample_weight=w_train))
print("Testing Performance")
print(classification_report(y_test, y_test_pred, sample_weight=w_test))

y_prob_train = rf.predict_proba(x_train)[:,1]
y_prob_test = rf.predict_proba(x_test)[:,1]
print(
    "Train AUC:",
    roc_auc_score(y_train, y_prob_train, sample_weight=w_train)
)
print(
    "Test AUC:",
    roc_auc_score(y_test, y_prob_test, sample_weight=w_test)
)

print("Confusion Matrix:")
print(confusion_matrix(
    y_test,
    y_test_pred,
    sample_weight=w_test
))

Training Performance
              precision    recall  f1-score   support

         0.0       0.88      0.82      0.85 314582919.0280208
         1.0       0.62      0.73      0.67 127414336.18678072

    accuracy                           0.80 441997255.21480155
   macro avg       0.75      0.78      0.76 441997255.21480155
weighted avg       0.81      0.80      0.80 441997255.21480155

Testing Performance
              precision    recall  f1-score   support

         0.0       0.82      0.75      0.78 77265524.38128349
         1.0       0.51      0.61      0.55 32505517.039818373

    accuracy                           0.71 109771041.42110187
   macro avg       0.66      0.68      0.67 109771041.42110187
weighted avg       0.73      0.71      0.72 109771041.42110187

Train AUC: 0.8625341533363995
Test AUC: 0.7513916933569769
Confusion Matrix:
[[57971585.69210383 19293938.68917945]
 [12661057.92747137 19844459.11234698]]


In [5]:
perm = permutation_importance(
    rf,
    x_test,
    y_test,
    scoring="roc_auc",
    n_repeats=30,
    random_state=42,
    sample_weight=w_test
)

importance = pd.DataFrame({
    "Feature": x_test.columns,
    "Importance": perm.importances_mean,
    "std": perm.importances_std
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance = importance[
    ~importance["Feature"].isin(["age", "sex"])
]

importance.head(5)


,Feature,Importance,std
10,vigorous_rec_activity,0.030383,0.006402
11,moderate_rec_activity,0.013529,0.003399
14,avg_drinks_per_day,0.013427,0.003823
6,fiber_g,0.003435,0.001575
2,energy_kcal,0.003384,0.001880
